# Tech Interview Pipeline — Independence and Correlation

A candidate goes through two interview rounds:

- Coding round with probability **0.7**
- System Design round with probability **0.5**

Assume the events are initially independent.

Questions:

1. What is the probability of clearing **both** rounds?
2. What is the probability of clearing **only one** round?
3. How does introducing **correlation** between the rounds change the results?

We will solve the problem analytically and then verify it through simulation.


## Step 1: Define the Events

Let:

- `C` = clears Coding round
- `S` = clears System Design round

Given:

$$P(C)=0.7$$

$$P(S)=0.5$$


In [1]:
p_coding = 0.7
p_design = 0.5

p_coding, p_design

(0.7, 0.5)

## Step 2: Probability of Clearing Both Rounds

Under independence:

$$P(C \cap S)=P(C)P(S)$$


In [2]:
p_both = p_coding * p_design
p_both

0.35

Therefore,

$$P(C \cap S)=0.7 \times 0.5 = 0.35$$

There is a **35%** chance of clearing both rounds.


## Step 3: Probability of Clearing Exactly One Round

Exactly one means:

- clears Coding but not Design
- clears Design but not Coding


In [3]:
p_only_coding = p_coding * (1 - p_design)
p_only_design = p_design * (1 - p_coding)
p_only_one = p_only_coding + p_only_design

p_only_coding, p_only_design, p_only_one

(0.35, 0.15000000000000002, 0.5)

Theoretical probabilities:

- Only Coding = 0.35
- Only Design = 0.15
- Only One = 0.50


In [4]:
p_neither = (1 - p_coding) * (1 - p_design)
p_neither

0.15000000000000002

In [5]:
p_both + p_only_one + p_neither

1.0

## Step 4: Simulate Independent Trials

In [6]:
import numpy as np
import pandas as pd

n = 10_000
rng = np.random.default_rng(42)

coding = rng.random(n) < p_coding
design = rng.random(n) < p_design

df_independent = pd.DataFrame({
    "Coding": coding,
    "Design": design,
})

df_independent.head()

,Coding,Design
0,False,False
1,True,False
2,False,True
3,True,True
4,True,True


In [7]:
both = (df_independent["Coding"] & df_independent["Design"]).mean()
only_one = (df_independent["Coding"] ^ df_independent["Design"]).mean()
neither = (
    (~df_independent["Coding"])
    & (~df_independent["Design"])
).mean()

pd.Series({
    "Both": both,
    "Only One": only_one,
    "Neither": neither,
})

Both        0.3476
Only One    0.5029
Neither     0.1495
dtype: float64

The simulated probabilities should be close to:

- Both ≈ 0.35
- Only One ≈ 0.50
- Neither ≈ 0.15


## Step 5: Simulate Positive Correlation

Suppose strong candidates tend to do well in both rounds.

One simple way to introduce positive correlation is to use the same underlying ability score.


In [8]:
ability = rng.random(n)

coding_corr = ability < 0.7
design_corr = ability < 0.5

df_correlated = pd.DataFrame({
    "Coding": coding_corr,
    "Design": design_corr,
})

df_correlated.head()

,Coding,Design
0,False,False
1,False,False
2,True,True
3,True,True
4,True,False


In [9]:
both_corr = (
    df_correlated["Coding"]
    & df_correlated["Design"]
).mean()

only_one_corr = (
    df_correlated["Coding"]
    ^ df_correlated["Design"]
).mean()

neither_corr = (
    (~df_correlated["Coding"])
    & (~df_correlated["Design"])
).mean()

pd.Series({
    "Both": both_corr,
    "Only One": only_one_corr,
    "Neither": neither_corr,
})

Both        0.4983
Only One    0.1952
Neither     0.3065
dtype: float64

## Discussion

Under independence:

$$P(C \cap S)=0.35$$

With positive correlation:

- The probability of clearing **both** rounds becomes larger.
- The probability of clearing **exactly one** round becomes smaller.
- Successes and failures tend to occur together.

The marginal probabilities remain fixed:

$$P(C)=0.7$$

$$P(S)=0.5$$

but the dependence structure changes the joint probabilities.


## Key Takeaways

1. Independence implies:

$$P(A \cap B)=P(A)P(B)$$

2. For independent events:

- Both = 0.35
- Only One = 0.50
- Neither = 0.15

3. Correlation changes the joint probabilities even when the marginal probabilities remain the same.
